In [ ]:
!pip install langchain
!pip install langchain-community
!pip install langchain-google-genai
!pip install sentence-transformers
!pip install faiss-cpu
!pip install pypdf
!pip install python-dotenv
!pip install transformers
!pip install accelerate
!pip install langchain_text_splitters
!pip install -U langchain-huggingface

**Loading the PDF**

In [3]:
from langchain_community.document_loaders import PyPDFLoader

pdfloader = PyPDFLoader('//content/Applied-Machine-Learning-and-AI-for-Engineers.pdf')

pages = pdfloader.load()

In [5]:
print(len(pages))

666


In [6]:
print(pages)

[Document(metadata={'producer': 'calibre (6.8.0) [https://calibre-ebook.com]', 'creator': 'calibre (6.8.0) [https://calibre-ebook.com]', 'creationdate': '2022-11-18T07:38:55+00:00', 'author': 'Jeff Prosise', 'moddate': '2022-11-18T15:38:56+08:00', 'title': 'Applied Machine Learning and AI for Engineers', 'source': '//content/Applied-Machine-Learning-and-AI-for-Engineers.pdf', 'total_pages': 666, 'page': 0, 'page_label': '1'}, page_content=''), Document(metadata={'producer': 'calibre (6.8.0) [https://calibre-ebook.com]', 'creator': 'calibre (6.8.0) [https://calibre-ebook.com]', 'creationdate': '2022-11-18T07:38:55+00:00', 'author': 'Jeff Prosise', 'moddate': '2022-11-18T15:38:56+08:00', 'title': 'Applied Machine Learning and AI for Engineers', 'source': '//content/Applied-Machine-Learning-and-AI-for-Engineers.pdf', 'total_pages': 666, 'page': 1, 'page_label': '2'}, page_content='1\nP r a i s e  f o r  A p p l i e d  M a c h i n e  L e a r n i n g  a n d  A I  f o r  E n g i n e e r s\nT

In [7]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)

docs = text_splitter.split_documents(pages)

In [ ]:
from langchain_huggingface.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')

In [9]:
from langchain_community.vectorstores import FAISS

vectorstore = FAISS.from_documents(docs, embeddings)

In [10]:
query = "What are neural networks?"

results = vectorstore.similarity_search(query, k=3)

for i, doc in enumerate(results):
    print("\n")
    print(f"Result {i+1}")
    print(doc.page_content[:500])



Result 1
speech into other languages, generate artwork and music, and
perform other tasks that were virtually impossible a few
years ago.
The multilayer perceptron is a simple neural network
comprising layers of neurons. Each neuron turns input into
output using a simple mathematical formula. Activation
functions further transform the data as it passes between
layers by introducing nonli nearities, enabling neural
networks to fit to a variety of datasets. Hidden layers
between the input layer and the out


Result 2
279
The simplest type of neural network is the multilayer
perceptron . It consists of nodes or neurons  arranged in
layers. The depth of the network is the number of layers; the
width is the number of neurons in each layer, which can be
different for every layer. State-of-the-art neural networks
sometimes contain 100 or more layers and thousands of neurons
in individual layers. A deep neural network  is one that
contains many layers, and it’s where the term deep learning
i

In [11]:
def retrieve_context(query,k=3):
  retrieved_docs = vectorstore.similarity_search(query, k=k)
  context = "\n".join([doc.page_content for doc in retrieved_docs])
  return context

In [13]:
from langchain_google_genai import ChatGoogleGenerativeAI
import os
from dotenv import load_dotenv

load_dotenv()  # Load environment variables from .env file

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", google_api_key=os.getenv("GEMINI_API_KEY"), temperature=0.3)

In [14]:
prompt_template = """
You are a Machine Learning, Deep Learning Knowledge and AI Assistant.

Answer the question ONLY using the provided context.

If the answer is not present in the context,
say:
"I could not find the answer in the provided document."

Context:
{context}

Question:
{question}

Answer:
"""

In [15]:
from langchain_core.prompts import PromptTemplate


prompt = PromptTemplate(template=prompt_template, input_variables=["context", "question"])

In [16]:
chain = prompt | llm

In [24]:
def generate_response(query):
  context = retrieve_context(query,3)
  response = chain.invoke({"context": context, "question": query})
  print("context")
  print(context)
  print('================')
  return response.content

In [26]:
generate_response("what is a transformer")

context
word context. A transformer knows that the word train  has
different meanings in “meet me at the train station” and
“it’s time to train a model,” and it factors word order
into its calculations.
BERT is a sophisticated transformer model installed with
language understanding when engineers at Google trained it
encoder-decoders for NMT, Transformer Encoder-
Decoders-Transformer Encoder-Decoders
Hugging Face, Using Pretrained Models to Classify
Text, Using Pretrained Models to Translate Text,
Building a BERT-Based Question Answering System-Fine-
Tuning BERT to Perform Sentiment Analysis
TransformerDecoder class, Building a Transformer-Based
NMT Model, Building a Transformer-Based NMT Model
TransformerEncoder class, Building a Transformer-Based
NMT Model, Building a Transformer-Based NMT Model
translate_text function, Building a Transformer-Based NMT
Model
translation (see neural machine translation)
simplicity itself.
Here’s an example that translates English to French:
 
from tra

'A transformer knows that a word can have different meanings based on its context and factors word order into its calculations. BERT is described as a sophisticated transformer model installed with language understanding.'

In [27]:
generate_response("what is Deep Learning") # k=3

context
comes the ability to do things in software that engineers
could only have dreamed about as recently as a decade
ago.
Over time, data scientists have devised special types of
neural networks that excel at certain tasks, including tasks
involving computer vision—for example, distilling
information from images—and tasks that involve human
languages such as translating English to French. We’ll take
a deep dive into neural networks beginning in Chapter 8, and
you’ll learn specifically how deep learning has elevated
machine learning to new heights.
S u p e r v i s e d  V e r s u s  U n s u p e r v i s e d  L e a r n i n g
Most ML models fall into one of two broad categories:
supervised learning  models and unsupervised learning  models.
The purpose of supervised learning models is to make
predictions. You train them with labeled data so that they
can take future inputs and predict what the labels will be.
learning. Deep learning  is machine learning performed with
neural networks. (T

"Deep learning is machine learning performed with neural networks. It is a subset of machine learning that relies primarily on neural networks, and most of what's considered AI today is accomplished with deep learning."

In [28]:
generate_response("what is overfitting in AI")

context
When you find that a neural network is fitting too tightly to
the training data, one way to combat overfitting and increase
the network’s ability to generalize is to reduce the
complexity of the network: reduce the number of layers, the
number of neurons in individual layers, or both. Another
328
accuracy and validation accuracy would be close to 0 in the
later stages of training a neural network. In the real world,
it rarely happens that way. Training accuracy for the model
in the previous section approached 100%, but validation
accuracy probably peaked between 80% and 85%. This means the
model isn’t generalizing as well as you’d like. It learned
the training data very well, but when presented with data it
hadn’t seen before (the validation data), it underperformed.
This may be a sign that the model is overfitting. In the end,
it’s not training accuracy that matters; it’s how
accurately the model responds to new data.
One way to combat overfitting is to reduce the depth of the

"Overfitting occurs when a neural network fits too tightly to the training data, learning it very well, but then underperforms when presented with data it hasn't seen before (validation data)."

In [29]:
generate_response("what is underfitting in AI")

context
328
accuracy and validation accuracy would be close to 0 in the
later stages of training a neural network. In the real world,
it rarely happens that way. Training accuracy for the model
in the previous section approached 100%, but validation
accuracy probably peaked between 80% and 85%. This means the
model isn’t generalizing as well as you’d like. It learned
the training data very well, but when presented with data it
hadn’t seen before (the validation data), it underperformed.
This may be a sign that the model is overfitting. In the end,
it’s not training accuracy that matters; it’s how
accurately the model responds to new data.
One way to combat overfitting is to reduce the depth of the
network, the width of individual layers, or both. Fewer
neurons means fewer trainable parameters, and fewer
parameters makes it harder for the network to fit too tightly
to the training data.
Another way to guard against overfitting is to introduce
When you find that a neural network is fitti

'I could not find the answer in the provided document.'

In [30]:
generate_response('who is sam altman')

context
658
A b o u t  t h e  A u t h o r
J e f f  P r o s i s e  is an engineer whose passion is to introduce
other engineers and software developers to the wonders of AI
and machine learning. Cofounder of Wintellect, he has written
nine books and hundreds of magazine articles, trained
thousands of developers at Microsoft, and spoken at some of
the world’s largest software conferences. In another life,
Jeff worked on high-powered laser systems and fusion-energy
research at Oak Ridge National Laboratory and Lawrence
Livermore National Laboratory. In his spare time, he builds
and flies large radio-control jets and goes out of his way to
get wet in the world’s best dive spots. Following the
acquisition of his company in 2021, Jeff serves as chief
learning officer at Atmosera, where he helps customers infuse
AI into their products.
reviewing chapters as I wrote them and providing constructive
feedback. They are engineers, mathematicians, data analysts,
and professors, and they happen to b

'I could not find the answer in the provided document.'